In [1]:
import pandas as pd

customers = pd.read_csv("../data/dim_customers.csv")
products = pd.read_csv("../data/dim_products.csv")
targets = pd.read_csv("../data/dim_targets_orders.csv")

order_line = pd.read_csv("../data/fact_order_line.csv")
aggregate = pd.read_csv("../data/fact_aggregate.csv")

In [2]:
order_line["order_placement_date"] = pd.to_datetime(
    order_line["order_placement_date"],
    dayfirst=True
)

order_line["agreed_delivery_date"] = pd.to_datetime(
    order_line["agreed_delivery_date"],
    dayfirst=True
)

order_line["actual_delivery_date"] = pd.to_datetime(
    order_line["actual_delivery_date"],
    dayfirst=True
)

aggregate["order_placement_date"] = pd.to_datetime(
    aggregate["order_placement_date"],
    dayfirst=True
)

print("Date columns converted successfully.")

Date columns converted successfully.


## Step 1

Merge the Order Line table with the Customer table.

This allows each order to include customer information such as:

- Customer Name
- City
- Currency


In [3]:
orders = order_line.merge(
    customers,
    on="customer_id",
    how="left"
)

print("Order Line merged with Customers successfully.")

Order Line merged with Customers successfully.


In [4]:
print("=" * 60)
print("Merged Dataset Shape")
print("=" * 60)

print(orders.shape)

Merged Dataset Shape
(24195, 14)


In [5]:
print("=" * 60)
print("First Five Rows")
print("=" * 60)

orders.head()

First Five Rows


,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,In Full,On Time,On Time In Full,customer_name,city,currency
0,FMR34203601,2025-03-01,789203,25891601,110,2025-03-04,2025-03-04,110,1,1,1,Rel Fresh,Vadodara,INR
1,FMR32320302,2025-03-01,789320,25891203,347,2025-03-02,2025-03-02,347,1,1,1,Whole Foods Market,"New Jersey, US",USD
2,FMR33320501,2025-03-01,789320,25891203,187,2025-03-03,2025-03-03,150,0,1,0,Whole Foods Market,"New Jersey, US",USD
3,FMR34220601,2025-03-01,789220,25891203,235,2025-03-04,2025-03-04,235,1,1,1,ShopRite,"New Jersey, US",USD
4,FMR33703603,2025-03-01,789703,25891203,176,2025-03-03,2025-03-03,176,1,1,1,Sorefoz Mart,Vadodara,INR


In [6]:
print(orders.shape)

(24195, 14)


## Step 2: Merge Product Information

The `order_line` table contains only the `product_id`.

To get detailed product information such as:

- Product Name
- Category
- Price (INR)
- Price (USD)

we merge the Products table with our current dataset.

In [7]:
orders = orders.merge(
    products,
    on="product_id",
    how="left"
)

print("Products merged successfully.")

Products merged successfully.


In [8]:
print("=" * 60)
print("Dataset Shape After Product Merge")
print("=" * 60)

print(orders.shape)

Dataset Shape After Product Merge
(24195, 18)


In [9]:
print("=" * 60)
print("First Five Records After Product Merge")
print("=" * 60)

orders.head()

First Five Records After Product Merge


,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,In Full,On Time,On Time In Full,customer_name,city,currency,product_name,category,price_INR,price_USD
0,FMR34203601,2025-03-01,789203,25891601,110,2025-03-04,2025-03-04,110,1,1,1,Rel Fresh,Vadodara,INR,AM Tea 500,beverages,225,6.0
1,FMR32320302,2025-03-01,789320,25891203,347,2025-03-02,2025-03-02,347,1,1,1,Whole Foods Market,"New Jersey, US",USD,AM Butter 500,Dairy,300,9.0
2,FMR33320501,2025-03-01,789320,25891203,187,2025-03-03,2025-03-03,150,0,1,0,Whole Foods Market,"New Jersey, US",USD,AM Butter 500,Dairy,300,9.0
3,FMR34220601,2025-03-01,789220,25891203,235,2025-03-04,2025-03-04,235,1,1,1,ShopRite,"New Jersey, US",USD,AM Butter 500,Dairy,300,9.0
4,FMR33703603,2025-03-01,789703,25891203,176,2025-03-03,2025-03-03,176,1,1,1,Sorefoz Mart,Vadodara,INR,AM Butter 500,Dairy,300,9.0


## Step 3: Verify Product Merge

Let's verify that product details have been merged correctly.

In [10]:
print("=" * 60)
print("Product Columns")
print("=" * 60)

print(
    orders[
        [
            "product_id",
            "product_name",
            "category",
            "price_INR",
            "price_USD"
        ]
    ].head()
)

Product Columns
   product_id   product_name   category  price_INR  price_USD
0    25891601     AM Tea 500  beverages        225        6.0
1    25891203  AM Butter 500      Dairy        300        9.0
2    25891203  AM Butter 500      Dairy        300        9.0
3    25891203  AM Butter 500      Dairy        300        9.0
4    25891203  AM Butter 500      Dairy        300        9.0


In [11]:
print(orders.shape)

(24195, 18)


## Step 4: Merge Order Performance Data

The Aggregate table contains order-level delivery performance.

The following metrics will be added:

- On Time
- In Full
- OTIF

These metrics are required for KPI calculations such as:

- On Time Delivery %
- In Full Delivery %
- OTIF %

In [12]:
orders = order_line.merge(
    customers,
    on="customer_id",
    how="left"
)

In [13]:
print("=" * 60)
print("Columns in Orders")
print("=" * 60)

print(orders.columns.tolist())

Columns in Orders
['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty', 'agreed_delivery_date', 'actual_delivery_date', 'delivery_qty', 'In Full', 'On Time', 'On Time In Full', 'customer_name', 'city', 'currency']


In [14]:
print(orders.columns.tolist())

['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty', 'agreed_delivery_date', 'actual_delivery_date', 'delivery_qty', 'In Full', 'On Time', 'On Time In Full', 'customer_name', 'city', 'currency']


In [15]:
orders = orders.merge(
    aggregate,
    on=["order_id", "customer_id", "order_placement_date"],
    how="left"
)

print("Aggregate table merged successfully.")

Aggregate table merged successfully.


In [16]:
orders = order_line.merge(
    customers,
    on="customer_id",
    how="left"
)

In [17]:
orders = orders.merge(
    products,
    on="product_id",
    how="left"
)

In [18]:
orders = orders.merge(
    aggregate,
    on=["order_id", "customer_id", "order_placement_date"],
    how="left"
)

print("Aggregate table merged successfully.")

Aggregate table merged successfully.


In [19]:
print("=" * 60)
print("Dataset Shape After Aggregate Merge")
print("=" * 60)

print(orders.shape)

Dataset Shape After Aggregate Merge
(24195, 21)


In [20]:
print("=" * 60)
print("Columns After Aggregate Merge")
print("=" * 60)

print(orders.columns.tolist())

Columns After Aggregate Merge
['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty', 'agreed_delivery_date', 'actual_delivery_date', 'delivery_qty', 'In Full', 'On Time', 'On Time In Full', 'customer_name', 'city', 'currency', 'product_name', 'category', 'price_INR', 'price_USD', 'on_time', 'in_full', 'otif']


In [21]:
print("=" * 60)
print("First Five Records")
print("=" * 60)

orders.head()

First Five Records


,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,In Full,On Time,...,customer_name,city,currency,product_name,category,price_INR,price_USD,on_time,in_full,otif
0,FMR34203601,2025-03-01,789203,25891601,110,2025-03-04,2025-03-04,110,1,1,...,Rel Fresh,Vadodara,INR,AM Tea 500,beverages,225,6.0,1,1,1
1,FMR32320302,2025-03-01,789320,25891203,347,2025-03-02,2025-03-02,347,1,1,...,Whole Foods Market,"New Jersey, US",USD,AM Butter 500,Dairy,300,9.0,1,1,1
2,FMR33320501,2025-03-01,789320,25891203,187,2025-03-03,2025-03-03,150,0,1,...,Whole Foods Market,"New Jersey, US",USD,AM Butter 500,Dairy,300,9.0,1,0,0
3,FMR34220601,2025-03-01,789220,25891203,235,2025-03-04,2025-03-04,235,1,1,...,ShopRite,"New Jersey, US",USD,AM Butter 500,Dairy,300,9.0,1,1,1
4,FMR33703603,2025-03-01,789703,25891203,176,2025-03-03,2025-03-03,176,1,1,...,Sorefoz Mart,Vadodara,INR,AM Butter 500,Dairy,300,9.0,1,1,1


## Step 5: Validate Delivery Performance Columns

Both the `fact_order_line` and `fact_aggregate` tables contain delivery performance metrics.

Before removing or keeping any columns, we verify whether they contain the same information.

In [22]:
print("=" * 60)
print("Comparing Delivery Performance Columns")
print("=" * 60)

print("In Full equal      :", (orders["In Full"] == orders["in_full"]).all())

print("On Time equal      :", (orders["On Time"] == orders["on_time"]).all())

print("OTIF equal         :", (orders["On Time In Full"] == orders["otif"]).all())

Comparing Delivery Performance Columns
In Full equal      : False
On Time equal      : True
OTIF equal         : False


## Step 6: Verify Aggregate Merge

After merging the Aggregate table, we verify that every order has matching order-level delivery metrics.

This ensures that the merge did not introduce missing values.


In [23]:
print("=" * 60)
print("Missing Values in Aggregate Columns")
print("=" * 60)

print(
    orders[
        [
            "on_time",
            "in_full",
            "otif"
        ]
    ].isnull().sum()
)

Missing Values in Aggregate Columns
on_time    0
in_full    0
otif       0
dtype: int64


# Step 7: Merge Customer Target Data

The target table contains the expected delivery performance for each customer.

These target values will later be compared against the actual delivery performance.

In [24]:
orders = orders.merge(
    targets,
    on="customer_id",
    how="left"
)

print("Customer target data merged successfully.")

Customer target data merged successfully.


In [25]:
print("=" * 60)
print("Dataset Shape After Target Merge")
print("=" * 60)

print(orders.shape)

Dataset Shape After Target Merge
(24195, 24)


In [26]:
print(targets.columns.tolist())

['customer_id', 'ontime_target%', 'infull_target%', 'otif_target%']


# Step 8: Validate Master Dataset

After merging all datasets, we validate the final master dataset by checking:

- Dataset dimensions
- Column names
- Missing values
- Data types

This ensures the dataset is ready for Exploratory Data Analysis (EDA) and KPI calculations.

In [27]:
print("=" * 60)
print("Master Dataset Information")
print("=" * 60)

orders.info()

Master Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 24195 entries, 0 to 24194
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   order_id              24195 non-null  str           
 1   order_placement_date  24195 non-null  datetime64[us]
 2   customer_id           24195 non-null  int64         
 3   product_id            24195 non-null  int64         
 4   order_qty             24195 non-null  int64         
 5   agreed_delivery_date  24195 non-null  datetime64[us]
 6   actual_delivery_date  24195 non-null  datetime64[us]
 7   delivery_qty          24195 non-null  int64         
 8   In Full               24195 non-null  int64         
 9   On Time               24195 non-null  int64         
 10  On Time In Full       24195 non-null  int64         
 11  customer_name         24195 non-null  str           
 12  city                  24195 non-null  str           
 13  

In [28]:
print("=" * 60)
print("Missing Values in Master Dataset")
print("=" * 60)

print(orders.isnull().sum())

Missing Values in Master Dataset
order_id                0
order_placement_date    0
customer_id             0
product_id              0
order_qty               0
agreed_delivery_date    0
actual_delivery_date    0
delivery_qty            0
In Full                 0
On Time                 0
On Time In Full         0
customer_name           0
city                    0
currency                0
product_name            0
category                0
price_INR               0
price_USD               0
on_time                 0
in_full                 0
otif                    0
ontime_target%          0
infull_target%          0
otif_target%            0
dtype: int64


In [29]:
print("=" * 60)
print("Final Columns in Master Dataset")
print("=" * 60)

for column in orders.columns:
    print(column)

Final Columns in Master Dataset
order_id
order_placement_date
customer_id
product_id
order_qty
agreed_delivery_date
actual_delivery_date
delivery_qty
In Full
On Time
On Time In Full
customer_name
city
currency
product_name
category
price_INR
price_USD
on_time
in_full
otif
ontime_target%
infull_target%
otif_target%


In [30]:
orders.to_csv("master_dataset.csv", index=False)

print("Master dataset saved successfully.")

Master dataset saved successfully.
